## Importaciones


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## 1) Carga del CSV


In [11]:
df_raw = pd.read_csv(
    "../data/raw/scraping_manual/alquiler_idealista.csv",
    delimiter=";",
    on_bad_lines="warn",
    encoding="utf-8",
    encoding_errors="replace",
    skipinitialspace=True,
    index_col=0
)

df_raw.head()

,tipo_inmueble,titulo,municipio,zona,precio_mensual_eur,habitaciones,superficie_m2,planta,exterior,ascensor,garaje_incluido,garaje_opcional,temporada,lujo,vistas_mar,amueblado,observaciones,Unnamed: 18
id,,,,,,,,,,,,,,,,,,
1,Piso,Piso en los Pinares,Santander,Los Castros,850,3,63,1,1,1,0,0,1,0,0,NaN,Curso escolar 2026-27 reformado,NaN
2,Piso,Piso Calleja Norte,Santander,El Sardinero,1400,3,107,3,1,1,1,0,0,0,1,NaN,Terraza vistas mar urbanización privada,NaN
3,Piso,Piso República de Colombia,Laredo,Zona Playa,600,1,58,9,1,1,0,0,0,0,0,1,Larga temporada terraza,NaN
4,Chalet independiente,Barrio de la Molina,Comillas,NaN,5000,6,392,NaN,NaN,NaN,1,0,1,1,0,NaN,Parcela 804 m2,NaN
5,Estudio,Espíritu Santo,Laredo,Centro,900,0,37,2,0,0,0,0,1,0,0,NaN,Vacacional sin ascensor,NaN


## 2) Quitar columna final vacía


In [12]:
df_sin_col_final = df_raw.copy()
df_sin_col_final = df_sin_col_final.iloc[:, :-1]


## 3) Chequeo de tipos (columnas clave)


In [13]:
df_sin_col_final[["precio_mensual_eur", "habitaciones", "superficie_m2"]].dtypes


precio_mensual_eur    int64
habitaciones          int64
superficie_m2         int64
dtype: object

## 4) Normalización de booleanos


In [14]:
df_bools_norm = df_sin_col_final.copy()

cols_bool = [
    "exterior",
    "ascensor",
    "garaje_incluido",
    "garaje_opcional",
    "temporada",
    "lujo",
    "vistas_mar",
    "amueblado",
]

df_bools_norm[cols_bool] = (
    df_bools_norm[cols_bool]
        .astype(str)
        .apply(lambda x: x.str.strip().str.lower())
        .replace({
            "sí": 1,
            "si": 1,
            "no": 0,
            "1": 1,
            "true": 1,
            "false": 0,
            "0": 0,
            "null": np.nan,
            "nan": np.nan,
        })
        .apply(pd.to_numeric, errors="coerce")
)


## 5) Validación rápida (amueblado)


In [15]:
df_bools_norm["amueblado"].value_counts(dropna=False)


amueblado
NaN    275
1.0    151
0.0     89
Name: count, dtype: int64

## 6) Exploración de planta


In [16]:
df_bools_norm["planta"].value_counts(dropna=False)


planta
NaN            91
2              63
3              53
1              52
1ª             47
4              30
5              30
Bajo           30
2ª             25
0              23
3ª             13
7              11
6               9
5ª              8
8               7
9               4
4ª              3
entreplanta     2
10              2
14              1
15              1
16              1
13              1
ático           1
12              1
Entreplanta     1
9ª              1
6ª              1
13ª             1
8ª              1
14ª             1
Name: count, dtype: int64

In [17]:
df_bools_norm["planta"].unique()


<StringArray>
[          '1',           '3',           '9',           nan,           '2',
           '4',           '7',           '0',           '5',           '6',
           '8', 'entreplanta',          '14',          '15',          '16',
          '13',       'ático',          '10',          '12',          '3ª',
          '5ª',          '2ª',        'Bajo',          '1ª', 'Entreplanta',
          '4ª',          '9ª',          '6ª',         '13ª',          '8ª',
         '14ª']
Length: 31, dtype: str

## 7) Limpieza de planta


In [18]:
df_planta_limpia = df_bools_norm.copy()

df_planta_limpia["planta"] = (
    df_planta_limpia["planta"]
        .astype(str)
        .str.lower()
        .str.strip()
)

df_planta_limpia["planta"] = df_planta_limpia["planta"].str.replace("ª", "", regex=False)

df_planta_limpia["planta"] = df_planta_limpia["planta"].replace({
    "bajo": 0,
    "entreplanta": 0.5,
    "atico": np.nan,
})

df_planta_limpia["planta"] = pd.to_numeric(df_planta_limpia["planta"], errors="coerce")

df_planta_limpia.loc[df_planta_limpia["planta"] > 20, "planta"] = np.nan
df_planta_limpia.loc[df_planta_limpia["tipo_inmueble"] == "casa", "planta"] = np.nan


## 8) Chequeo post-limpieza (planta)


In [19]:
df_planta_limpia["planta"].unique()


array([ 1. ,  3. ,  9. ,  nan,  2. ,  4. ,  7. ,  0. ,  5. ,  6. ,  8. ,
        0.5, 14. , 15. , 16. , 13. , 10. , 12. ])

In [20]:
df_planta_limpia["planta"].describe()


count    423.000000
mean       2.741135
std        2.537120
min        0.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       16.000000
Name: planta, dtype: float64

## 9) Valores de tipo_inmueble


In [21]:
df_planta_limpia["tipo_inmueble"].unique()


<StringArray>
[                       'Piso',        'Chalet independiente',
                     'Estudio',              'Chalet pareado',
                       'Ático',                      'Dúplex',
              'Chalet adosado',          'Casa independiente',
                      'Chalet',               'Finca rústica',
                     'Caserón',                  'Casa rural',
              'Casa de pueblo', 'Casa o chalet independiente']
Length: 14, dtype: str

## 10) Tipo (piso/casa) + subtipo


In [22]:
df_tipo_subtipo = df_planta_limpia.copy()

df_tipo_subtipo["tipo_inmueble"] = (
    df_tipo_subtipo["tipo_inmueble"]
      .astype(str)
      .str.strip()
      .str.lower()
)

df_tipo_subtipo["subtipo"] = df_tipo_subtipo["tipo_inmueble"].replace({"nan": np.nan})

set_piso = {"piso", "estudio", "ático", "atico", "dúplex", "duplex"}
df_tipo_subtipo["tipo_inmueble"] = np.where(df_tipo_subtipo["tipo_inmueble"].isin(set_piso), "piso", "casa")

df_tipo_subtipo["subtipo"] = df_tipo_subtipo["subtipo"].replace({
    "atico": "ático",
    "duplex": "dúplex",
})

mask_piso = df_tipo_subtipo["tipo_inmueble"].eq("piso")
mask_subtipo_vacio = df_tipo_subtipo["subtipo"].isna() | df_tipo_subtipo["subtipo"].eq("piso")
df_tipo_subtipo.loc[mask_piso & mask_subtipo_vacio, "subtipo"] = "apartamento"

mask_casa = df_tipo_subtipo["tipo_inmueble"].eq("casa")
df_tipo_subtipo.loc[mask_casa & df_tipo_subtipo["subtipo"].isna(), "subtipo"] = "casa independiente"


## 11) Quitar rústicas


In [23]:
df_sin_rustica = df_tipo_subtipo.copy()

valores_eliminar = ["Finca rústica", "finca rústica"]
df_sin_rustica = df_sin_rustica[~df_sin_rustica["subtipo"].isin(valores_eliminar)]


## 12) Homogeneizar subtipo


In [24]:
df_subtipo_homog = df_sin_rustica.copy()

df_subtipo_homog["subtipo"] = (
    df_subtipo_homog["subtipo"]
        .str.lower()
        .str.strip()
        .replace({
            "chalét independiente": "casa independiente",
            "chalet independiente": "casa independiente",
            "chalét adosado": "casa adosada",
            "chalet adosado": "casa adosada",
            "chalét pareado": "casa adosada",
            "chalet pareado": "casa adosada",
            "casa o chalet independiente": "casa independiente",
            "chalet": "casa independiente",
        })
)


## 13) Conteos


In [25]:
df_subtipo_homog["tipo_inmueble"].value_counts(dropna=False)


tipo_inmueble
piso    428
casa     86
Name: count, dtype: int64

In [26]:
df_subtipo_homog["subtipo"].value_counts(dropna=False)


subtipo
apartamento           370
casa independiente     43
casa adosada           32
ático                  28
dúplex                 16
estudio                14
casa rural              7
casa de pueblo          3
caserón                 1
Name: count, dtype: int64

## 14) One-hot


In [27]:
df_onehot = df_subtipo_homog.copy()

df_onehot = pd.get_dummies(
    df_onehot,
    columns=["tipo_inmueble", "subtipo"],
    drop_first=True
)

df_onehot.columns


Index(['titulo', 'municipio', 'zona', 'precio_mensual_eur', 'habitaciones',
       'superficie_m2', 'planta', 'exterior', 'ascensor', 'garaje_incluido',
       'garaje_opcional', 'temporada', 'lujo', 'vistas_mar', 'amueblado',
       'observaciones', 'tipo_inmueble_piso', 'subtipo_casa adosada',
       'subtipo_casa de pueblo', 'subtipo_casa independiente',
       'subtipo_casa rural', 'subtipo_caserón', 'subtipo_dúplex',
       'subtipo_estudio', 'subtipo_ático'],
      dtype='str')

## 15) Flag vacacional (texto)


In [28]:
df_flags_texto = df_onehot.copy()

df_flags_texto["observaciones"] = (
    df_flags_texto["observaciones"]
        .astype(str)
        .str.lower()
        .str.strip()
)

patron_verano = r"\b(quincena|quincenas|verano|julio|agosto)\b"
patron_junio = r"\bjunio\b"
patron_excluir = r"(sep|sept|septiembre|hasta.*junio|temporal\s+sep|contrato temporal|larga duraci[oó]n)"

cond_verano_fuerte = df_flags_texto["observaciones"].str.contains(patron_verano, regex=True, na=False)
cond_junio = df_flags_texto["observaciones"].str.contains(patron_junio, regex=True, na=False)
cond_excluir = df_flags_texto["observaciones"].str.contains(patron_excluir, regex=True, na=False)

df_flags_texto["es_vacacional_verano"] = np.where(
    (cond_verano_fuerte | cond_junio) & (~cond_excluir),
    1,
    0
)


C:\Users\pdelr\AppData\Local\Temp\ipykernel_19276\1875211171.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cond_verano_fuerte = df_flags_texto["observaciones"].str.contains(patron_verano, regex=True, na=False)
C:\Users\pdelr\AppData\Local\Temp\ipykernel_19276\1875211171.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cond_excluir = df_flags_texto["observaciones"].str.contains(patron_excluir, regex=True, na=False)


## 16) Muestras (validación rápida)


In [29]:
df_flags_texto[df_flags_texto["es_vacacional_verano"] == 1][["observaciones"]].head(20)


,observaciones
id,
17,quincenas julio-agosto
23,no agosto parcial
26,agosto 3000 quincena
72,temporada julio/agosto lujo
77,temporada quincenas julio
81,verano quincenas rebaja 1900->1700 vistas mar
82,verano quincenas rebaja 1200->1100
83,verano reformado amueblado sin ascensor
84,julio 3300 agosto 3800 (temporal)


In [30]:
df_flags_texto[df_flags_texto["observaciones"].str.contains("verano|julio|agosto", na=False)][["observaciones"]]


,observaciones
id,
17,quincenas julio-agosto
23,no agosto parcial
26,agosto 3000 quincena
72,temporada julio/agosto lujo
77,temporada quincenas julio
81,verano quincenas rebaja 1900->1700 vistas mar
82,verano quincenas rebaja 1200->1100
83,verano reformado amueblado sin ascensor
84,julio 3300 agosto 3800 (temporal)


## 17) Preview final


In [31]:
df_flags_texto.head(20)


,titulo,municipio,zona,precio_mensual_eur,habitaciones,superficie_m2,planta,exterior,ascensor,garaje_incluido,...,tipo_inmueble_piso,subtipo_casa adosada,subtipo_casa de pueblo,subtipo_casa independiente,subtipo_casa rural,subtipo_caserón,subtipo_dúplex,subtipo_estudio,subtipo_ático,es_vacacional_verano
id,,,,,,,,,,,,,,,,,,,,,
1,Piso en los Pinares,Santander,Los Castros,850,3,63,1.0,1.0,1.0,0,...,True,False,False,False,False,False,False,False,False,0
2,Piso Calleja Norte,Santander,El Sardinero,1400,3,107,3.0,1.0,1.0,1,...,True,False,False,False,False,False,False,False,False,0
3,Piso República de Colombia,Laredo,Zona Playa,600,1,58,9.0,1.0,1.0,0,...,True,False,False,False,False,False,False,False,False,0
4,Barrio de la Molina,Comillas,NaN,5000,6,392,NaN,NaN,NaN,1,...,False,False,False,True,False,False,False,False,False,0
5,Espíritu Santo,Laredo,Centro,900,0,37,2.0,0.0,0.0,0,...,True,False,False,False,False,False,False,True,False,0
6,Polanco,Polanco,NaN,1750,4,150,NaN,NaN,NaN,1,...,False,True,False,False,False,False,False,False,False,0
7,Puerto Chico,Santander,Puerto Chico,575,1,50,4.0,NaN,NaN,0,...,True,False,False,False,False,False,False,False,True,0
8,Luis Buñuel,Santander,Peñacastillo,1300,3,150,NaN,NaN,NaN,1,...,False,True,False,False,False,False,False,False,False,0
9,San Andrés,Castro-Urdiales,Cotolino,625,1,47,1.0,1.0,1.0,0,...,True,False,False,False,False,False,False,False,False,0
